---
title: Daily precipitation climatology and the annual march of the rain belt (GPM IMERG)
author: Harsha R. Hampapura
date: 2026-09-30
---

## Daily precipitation climatology: the annual march of the rain belt

[GPM IMERG Final (daily)](https://gdex.ucar.edu/datasets/d734000/) is a global, 0.1°
estimate of **daily** precipitation from NASA's satellite-and-gauge record. **GPM** —
the [Global Precipitation Measurement](https://climatedataguide.ucar.edu/climate-data/gpm-global-precipitation-measurement-mission)
mission — and **IMERG** (*Integrated Multi-satellitE Retrievals for GPM*) merge
microwave and infrared observations into one daily rain field; GDEX's copy spans
**June 2000 – September 2025**.

This notebook reproduces NASA's *IMERG Daily Climatology* animation from GDEX data. We

1. average every calendar day across recent years (2020–2025) into a **day-of-year climatology**,
2. smooth it with a **30-day trailing average** to remove the noise of isolated
   heavy-rain days, then
3. animate the 366-day cycle to watch the **Intertropical Convergence Zone (ITCZ)** —
   the equatorial band of heavy tropical rain — and the monsoons march north and south
   with the seasons.

Compared with the [monthly climatology notebook](./imerg_monthly_itcz.ipynb), the daily
record resolves seasonal transitions — monsoon onset, the ITCZ's migration — far more
smoothly.

:::{seealso} Original visualization
This workflow reproduces **"IMERG Daily Climatology"** — NASA's Scientific Visualization
Studio, [SVS #4759](https://svs.gsfc.nasa.gov/4759/) (released 15 October 2019).
Visualizer: Alex Kekesi (GST, Inc.); data visualizer: Horace Mitchell (NASA/GSFC);
scientists: George Huffman and Dalia B. Kirschbaum (NASA/GSFC); producers: Ryan
Fitzgibbons and Joy Ng (USRA). *Credit: NASA's Scientific Visualization Studio.* Our
version uses the GDEX IMERG daily record (2000–2025) rather than the 2001–2018 span in
the original.
:::

In [1]:
import os
import shutil
import calendar
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client, LocalCluster
import datetime

## Open the dataset

IMERG daily is distributed as one NetCDF granule per day. GDEX adds a **kerchunk**
reference so a single `xr.open_dataset(..., engine="kerchunk")` call exposes the whole
~25-year daily record as one virtual `xarray.Dataset`.

The cell below auto-selects the access path: the POSIX reference where `/gdex/data` is
mounted (NCAR HPC), otherwise the OSDF reference streamed over the network. The OSDF
form's chunk URLs use the `osdf://` scheme, resolved by
[pelicanfs](https://github.com/PelicanPlatform/pelicanfs).

In [2]:
# POSIX reference where the GDEX data mount is present (NCAR HPC; its chunks live
# under /glade/campaign), otherwise the OSDF-routed reference over the network.
posix_ref = "/gdex/data/d734000/kerchunk/GPM_3IMERGDF_v07.json"
osdf_ref  = "https://data.gdex.ucar.edu/d734000/kerchunk/GPM_3IMERGDF_v07-remote-osdf.json"

kerchunk_ref = posix_ref if os.path.exists(posix_ref) else osdf_ref
print(f"Access: {'POSIX mount' if kerchunk_ref == posix_ref else 'OSDF (remote)'}")
print(kerchunk_ref)

Access: POSIX mount
/gdex/data/d734000/kerchunk/GPM_3IMERGDF_v07.json


## Set up a Dask cluster

Building the day-of-year climatology reads **several recent years of the daily record**, so we
farm the work out to a [Dask](https://docs.dask.org/) cluster. As in the other GDEX
examples, the cluster is chosen automatically by environment:

- **Casper / Derecho** (a PBS scheduler is present) → a `PBSCluster` spreads the read
  across scheduler-managed workers.
- **Laptop / CIRRUS Binder** (no PBS) → a `LocalCluster` uses the local cores.

In [3]:
def get_pbs_cluster(n_workers=10):
    """PBSCluster on NCAR HPC (Casper/Derecho)."""
    scratch = f"/glade/derecho/scratch/{os.environ.get('USER', '')}"
    cluster = PBSCluster(
        job_name        = "imerg-daily",
        cores           = 1,
        memory          = "10GiB",
        processes       = 1,
        local_directory = f"{scratch}/dask/spill/",
        log_directory   = f"{scratch}/dask/logs/",
        resource_spec   = "select=1:ncpus=1:mem=10GB",
        queue           = "casper",
        walltime        = "2:00:00",
        interface       = "ext",
    )
    cluster.scale(n_workers)
    return cluster


def get_local_cluster():
    """LocalCluster for a laptop or a CIRRUS Binder pod (no PBS scheduler)."""
    return LocalCluster()

:::{important} No edits needed
This cell **automatically selects the right Dask cluster** — a PBS cluster on NCAR HPC
(Casper/Derecho), or a `LocalCluster` on CIRRUS Binder or a laptop. **You don't need to
change anything;** the helper functions above show how each is built.
:::

In [4]:
on_hpc  = bool(shutil.which("qsub"))
cluster = get_pbs_cluster() if on_hpc else get_local_cluster()
client  = Client(cluster)
if on_hpc:
    client.wait_for_workers(n_workers=8)

print(f"Detected {'NCAR HPC -> PBSCluster' if on_hpc else 'no PBS -> LocalCluster'}")
cluster

Detected NCAR HPC -> PBSCluster


PBSCluster(9085433b, 'tcp://128.117.208.177:41515', workers=10, threads=10, memory=100.00 GiB)

2026-09-30 12:31:05,722 - distributed.scheduler - ERROR - Couldn't gather keys: {'finalize-hlgfinalizecompute-30cd69f0222f40a2a184bd708ef8e229': 'waiting'}


## Day-of-year climatology

We average each calendar day (day-of-year 1–366) across **recent years (2020–2025)** to get
the mean annual cycle at daily resolution. Precipitation is already in **mm/day**, so no unit
conversion is needed; we mask the fill value by keeping non-negative rates.

:::{warning} A large, memory-aware reduction
Even a few years of daily files is a big read, and the store's native **1-day chunking**
would build a task graph large enough to overwhelm the `dask-jobqueue` scheduler (which runs
in this kernel). Two settings keep it tractable: open with **large time chunks**
(`chunks={"time": 120}`) to shrink the graph, and let
**[flox](https://flox.readthedocs.io/)** perform the group-by reduction
(`method="cohorts"`, optimal for day-of-year climatologies). We also coarsen 0.1° → 0.4° to
lighten the reduction and keep the animation small — widen the `time` slice or the coarsening
factor to trade coverage for speed.
:::

In [5]:
%%time
xr.set_options(use_flox=True)

# Open with LARGE time chunks, not the native 1-day chunks: shrinks the task
# graph ~10x. This matters because dask-jobqueue's scheduler lives in THIS
# kernel, and the native-chunked groupby builds an ~84k-task graph that kills it.
ds = xr.open_dataset(kerchunk_ref, engine="kerchunk", chunks={"time": 120})
precip = ds["precipitation"].where(ds["precipitation"] >= 0)     # mm/day; mask fill
precip = precip.sel(time=slice("2020", "2025"))
precip = precip.coarsen(lon=4, lat=4, boundary="trim").mean()    # 0.1 -> 0.4 deg

# 'cohorts' is flox's memory-optimal method for day-of-year climatologies.
doy_clim = precip.groupby("time.dayofyear").mean("time", method="cohorts")
doy_clim = doy_clim.transpose("dayofyear", "lat", "lon").compute()
doy_clim.attrs["units"] = "mm/day"
print("shape:", doy_clim.shape)
doy_clim

shape: (366, 450, 900)
CPU times: user 45.2 s, sys: 4.51 s, total: 49.7 s
Wall time: 2min 32s


<xarray.DataArray 'precipitation' (dayofyear: 366, lat: 450, lon: 900)> Size: 593MB
array([[[0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        ...,
        [0.13963543, 0.1316146 , 0.12057292, ..., 0.15562502,
         0.15473959, 0.15052085],
        [0.10130156, 0.09093682, 0.08094483, ..., 0.11298958,
         0.11608555, 0.11051533],
        [0.1462941 , 0.15371494, 0.15617836, ..., 0.13316096,
         0.13986361, 0.1372768 ]],

       [[0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
...
        [0.0045    , 0.0045    , 0.0045    , ..., 0.0055    ,
         0.0055    , 0.0055    ],
        [0.0155    , 0.0148125 , 0.01575   , ..., 0.00874202,
         0.0157859 , 0.01816755],
        [0.07299999, 0.07813104, 0.05849798, ..., 0.0645761 ,
         0.07070376, 0.07031532]],

       [[       nan,        nan,        nan, ...,        nan,
                nan,        nan],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        ...,
        [0.5665791 , 0.5355457 , 0.50514746, ..., 0.71641004,
         0.6827652 , 0.5915272 ],
        [0.31023717, 0.30230558, 0.25339276, ..., 0.28093106,
         0.29972145, 0.28824568],
        [0.02543787, 0.02543787, 0.01512931, ..., 0.0357786 ,
         0.02785988, 0.0251601 ]]], dtype=float32)
Coordinates:
  * dayofyear  (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
  * lat        (lat) float64 4kB -89.8 -89.4 -89.0 -88.6 ... 88.6 89.0 89.4 89.8
  * lon        (lon) float32 4kB -179.8 -179.4 -179.0 ... 179.0 179.4 179.8
Attributes:
    units:      mm/day
    long_name:  Daily mean precipitation rate (combined microwave-IR) estimat...

## Smooth with a 30-day trailing average

Individual calendar days are noisy — one big storm in a single year leaves a spot in that
day's mean. Following NASA's visualization, we apply a **30-day trailing average** over the
day-of-year axis (wrapping across the December–January boundary) to reveal the smooth
seasonal march.

In [7]:
# 30-day trailing mean over day-of-year, wrapping across the year boundary.
n = 30
wrapped = xr.concat(
    [doy_clim.isel(dayofyear=slice(-(n - 1), None)), doy_clim], dim="dayofyear")

smooth = (wrapped.rolling(dayofyear=n, center=False).mean().isel(dayofyear=slice(n - 1, None)))
smooth["dayofyear"] = doy_clim["dayofyear"]
smooth.attrs["units"] = "mm/day"

print("smoothed range:", round(float(smooth.min()), 2), "-",
      round(float(smooth.max()), 2), "mm/day")
smooth

smoothed range: 0.0 - 50.78 mm/day


<xarray.DataArray 'precipitation' (dayofyear: 366, lat: 450, lon: 900)> Size: 593MB
array([[[       nan,        nan,        nan, ...,        nan,
                nan,        nan],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        ...,
        [0.05693751, 0.05465646, 0.05247932, ..., 0.0627089 ,
         0.06115375, 0.05755411],
        [0.05392698, 0.05268494, 0.05094295, ..., 0.05393939,
         0.05477588, 0.05478062],
        [0.06375505, 0.06306828, 0.06487417, ..., 0.06316096,
         0.06443425, 0.06521252]],

       [[       nan,        nan,        nan, ...,        nan,
                nan,        nan],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
...
        [0.03347411, 0.03246362, 0.03162198, ..., 0.03364107,
         0.03324525, 0.03284209],
        [0.04026296, 0.03959489, 0.03985474, ..., 0.04091257,
         0.04102794, 0.04156286],
        [0.0592653 , 0.05854149, 0.06036668, ..., 0.05895869,
         0.06033777, 0.06158486]],

       [[       nan,        nan,        nan, ...,        nan,
                nan,        nan],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        [0.        , 0.        , 0.        , ..., 0.        ,
         0.        , 0.        ],
        ...,
        [0.05236008, 0.05031514, 0.04846023, ..., 0.0575214 ,
         0.05600409, 0.05255967],
        [0.05055026, 0.04965371, 0.04824479, ..., 0.05017307,
         0.05090636, 0.05109677],
        [0.05887858, 0.05794444, 0.05966823, ..., 0.05872227,
         0.05977213, 0.06063662]]], dtype=float32)
Coordinates:
  * dayofyear  (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
  * lat        (lat) float64 4kB -89.8 -89.4 -89.0 -88.6 ... 88.6 89.0 89.4 89.8
  * lon        (lon) float32 4kB -179.8 -179.4 -179.0 ... 179.0 179.4 179.8
Attributes:
    units:      mm/day
    long_name:  Daily mean precipitation rate (combined microwave-IR) estimat...

## Animate the annual cycle

Playing the smoothed daily fields as a loop (subsampled to keep the GIF small) reveals the **ITCZ** and the monsoons
marching north and south through the year — the same story as the monthly notebook, but
resolved day by day. Following NASA's visualization, cooler colors are drier and warmer
colors wetter (mm/day).

In [ ]:
%%time
lat, lon = smooth["lat"], smooth["lon"]
frames = range(0, smooth.sizes["dayofyear"], 3)   # every 3rd day keeps the GIF small

fig, ax = plt.subplots(figsize=(8, 4.2),
                       subplot_kw={"projection": ccrs.PlateCarree()},
                       constrained_layout=True)
ax.set_global()
ax.coastlines(linewidth=0.4, color="0.5")

mesh = ax.pcolormesh(lon, lat, smooth.isel(dayofyear=0),
                     transform=ccrs.PlateCarree(), cmap="turbo",
                     vmin=0, vmax=25, shading="auto", rasterized=True)
cb = plt.colorbar(mesh, ax=ax, orientation="horizontal", shrink=0.7, pad=0.05, extend="max")
cb.set_label("Precipitation (mm/day)")
title = ax.set_title("")

def _label(doy):   # day-of-year -> "Mon DD" (2016 is a leap year -> 366 days)
    return (datetime.date(2016, 1, 1) + datetime.timedelta(days=int(doy) - 1)).strftime("%b %d")

def update(i):
    mesh.set_array(smooth.isel(dayofyear=i).values.ravel())
    title.set_text(f"IMERG daily precipitation climatology (30-day rolling mean) - {_label(smooth.dayofyear.values[i])}")
    return mesh, title

anim = animation.FuncAnimation(fig, update, frames=frames, interval=120)
anim.save("imerg_daily_climatology.gif", writer=animation.PillowWriter(fps=10), dpi=70)
plt.close(fig)

from IPython.display import Image
Image("imerg_daily_climatology.gif")